# JAX-Accelerated MLP for Feynman Symbolic Regression

This notebook demonstrates training a simple, fast Multi-Layer Perceptron (MLP) in pure JAX to perform symbolic regression on the **Relativistic Time Dilation** equation ($I.15.3t$) from the AI Feynman database:

$$t' = \frac{t}{\sqrt{1 - \frac{v^2}{c^2}}}$$

We will:
1. Generate a dataset in under a millisecond using our JAX-accelerated `FeynmanDatasetGenerator`.
2. Implement a pure JAX MLP with SiLU activation functions.
3. Train the model using the `optax` Adam optimizer.
4. Benchmark the speed and evaluate regression accuracy with Matplotlib visualizations.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../"))

import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
import time
from eigenflow.datasets import FeynmanDatasetGenerator

# Set up JAX keys
key = jax.random.PRNGKey(42)
dataset_key, init_key = jax.random.split(key)

# 1. Initialize dataset generator for Relativistic Time Dilation
generator = FeynmanDatasetGenerator("I.15.3t")

# 2. Generate 20,000 samples with input standardized scaling and 1% noise
X, y, metadata = generator.generate(
    key=dataset_key,
    num_samples=20000,
    noise_level=0.01,
    noise_type="gaussian",
    input_scaling="standardize",
    target_scaling="raw"
)

print(f"Generated {X.shape[0]} samples with {X.shape[1]} input variables.")
print(f"Formula: {metadata['formula']}")
print(f"Variables: {metadata['variables']}")

In [ ]:
# --- 1. MLP Architecture & Helpers ---

# Xavier/Glorot Initialization for parameters
def init_mlp_params(rng, layer_sizes):
    keys = jax.random.split(rng, len(layer_sizes) - 1)
    params = []
    for i in range(len(layer_sizes) - 1):
        in_dim = layer_sizes[i]
        out_dim = layer_sizes[i+1]
        w_key, b_key = jax.random.split(keys[i])
        lim = jnp.sqrt(6.0 / (in_dim + out_dim))
        w = jax.random.uniform(w_key, shape=(in_dim, out_dim), minval=-lim, maxval=lim)
        b = jnp.zeros((out_dim,))
        params.append((w, b))
    return params

# Forward pass with SiLU activation
def forward_mlp(params, X):
    activation = X
    for w, b in params[:-1]:
        activation = jax.nn.silu(jnp.dot(activation, w) + b)
    w_last, b_last = params[-1]
    return jnp.squeeze(jnp.dot(activation, w_last) + b_last) # squeeze to shape (N,)

# Mean Squared Error Loss
def mse_loss(params, X, y):
    preds = forward_mlp(params, X)
    return jnp.mean((preds - y) ** 2)

# --- 2. Model Initialization & Setup ---

# Define layer sizes: 3 inputs (t, v, c) -> 64 -> 64 -> 1 output (t')
layer_sizes = [3, 64, 64, 1]
params = init_mlp_params(init_key, layer_sizes)

# Setup Optimizer
lr = 3e-3
optimizer = optax.adam(lr)
opt_state = optimizer.init(params)

# Compiled update step
@jax.jit
def train_step(params, opt_state, X_batch, y_batch):
    loss, grads = jax.value_and_grad(mse_loss)(params, X_batch, y_batch)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Split train/test sets (80-20 split)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# --- 3. Training Loop ---

epochs = 5000
losses = []
test_losses = []

print("Starting training...")
start_time = time.time()

# Run training
for epoch in range(epochs):
    params, opt_state, loss = train_step(params, opt_state, X_train, y_train)
    losses.append(loss)
    
    if epoch % 500 == 0 or epoch == epochs - 1:
        test_loss = mse_loss(params, X_test, y_test)
        test_losses.append((epoch, float(test_loss)))
        print(f"Epoch {epoch:4d} | Train Loss: {loss:.6f} | Test Loss: {test_loss:.6f}")

end_time = time.time()
total_time = end_time - start_time
print(f"\nTraining completed in: {total_time:.4f} seconds!")
print(f"Time per epoch: {total_time/epochs*1000:.4f} milliseconds.")

# --- 4. Plot Results ---

preds_test = forward_mlp(params, X_test)
raw_test_inputs = metadata["X_raw"][split:]
raw_test_targets = metadata["y_raw"][split:]

from eigenflow.utils import plot_regression_results

plot_regression_results(
    losses=losses,
    y_true=y_test,
    y_pred=preds_test,
    test_losses=test_losses,
    X_raw=raw_test_inputs,
    y_raw=raw_test_targets,
    variables=metadata["variables"],
    slice_feature_idx=1,
    model_label="JAX MLP Prediction",
    title="MLP Relativistic Time Dilation Regression (JAX)"
)